# 04 · Baselines — naive, seasonal naive, moving average, Prophet
Every fancy model must beat these. All models forecast the same 28-day test window and are scored with
**WAPE** and **RMSSE** (M5 metrics).

In [1]:
import sys; sys.path.insert(0, "..")
import warnings, logging; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, json
from src.config import *
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": .3})

from src.features import time_split
from src.metrics import wape, rmsse
feat = pd.read_parquet(PROC_DIR / "features.parquet")
_, va, te = time_split(feat)
hist = feat[feat.date < te.date.min()][["id", "date", "sales"]]       # everything before the test window
test = te[["id", "date", "sales"]].copy()
N_PROPHET = 40          # Prophet fits one model per series -> use the top-N series (increase on real data/time)

## Naive, seasonal naive (same weekday last week), 28-day moving average

In [2]:
def last_values(g, h=HORIZON):
    v = g.sales.values
    return pd.Series({"naive": np.repeat(v[-1], h), "snaive": np.resize(v[-7:], h), "ma28": np.repeat(v[-28:].mean(), h)})

rows = []
for sid, g in hist.groupby("id"):
    p = last_values(g)
    d = test[test.id == sid].sort_values("date")
    rows.append(pd.DataFrame({"id": sid, "date": d.date.values, "pred_naive": p["naive"],
                              "pred_snaive": p["snaive"], "pred_ma28": p["ma28"]}))
base = test.merge(pd.concat(rows), on=["id", "date"])

## Prophet (weekly + yearly seasonality + event days) on the top-N series

In [3]:
from prophet import Prophet
logging.getLogger("cmdstanpy").setLevel(logging.ERROR); logging.getLogger("prophet").setLevel(logging.ERROR)
events = (feat.loc[feat.event_name_1.notna(), ["date", "event_name_1"]].drop_duplicates("date")
          .rename(columns={"date": "ds", "event_name_1": "holiday"}))
top_ids = hist.groupby("id")["sales"].sum().nlargest(N_PROPHET).index
prophet_rows = []
for sid in top_ids:
    g = hist[hist.id == sid].rename(columns={"date": "ds", "sales": "y"})
    m = Prophet(weekly_seasonality=True, yearly_seasonality=True, daily_seasonality=False,
                holidays=events, interval_width=0.8)
    m.fit(g[["ds", "y"]])
    d = test[test.id == sid].sort_values("date")
    fc = m.predict(pd.DataFrame({"ds": d.date.values}))
    prophet_rows.append(pd.DataFrame({"id": sid, "date": d.date.values, "pred_prophet": fc.yhat.clip(lower=0).values}))
base = base.merge(pd.concat(prophet_rows), on=["id", "date"], how="left")
base.to_parquet(OUT_DIR / "pred_baselines.parquet")

19:29:50 - cmdstanpy - INFO - Chain [1] start processing


19:29:50 - cmdstanpy - INFO - Chain [1] done processing


19:29:50 - cmdstanpy - INFO - Chain [1] start processing


19:29:50 - cmdstanpy - INFO - Chain [1] done processing


19:29:50 - cmdstanpy - INFO - Chain [1] start processing


...

19:29:59 - cmdstanpy - INFO - Chain [1] done processing


19:29:59 - cmdstanpy - INFO - Chain [1] start processing


19:29:59 - cmdstanpy - INFO - Chain [1] done processing


19:29:59 - cmdstanpy - INFO - Chain [1] start processing


19:29:59 - cmdstanpy - INFO - Chain [1] done processing


## Scores
Prophet is scored on its N series; the other baselines on both all series and the same N series.

In [4]:
def score(sub, col, train):
    return dict(model=col.replace("pred_", ""), n_series=sub.id.nunique(),
                WAPE=wape(sub.sales, sub[col]), RMSSE=rmsse(train[train.id.isin(sub.id)], sub, col))
res = []
for col in ["pred_naive", "pred_snaive", "pred_ma28"]:
    res.append({**score(base, col, hist), "scope": "all series"})
sub = base[base.id.isin(top_ids)]
for col in ["pred_naive", "pred_snaive", "pred_ma28", "pred_prophet"]:
    res.append({**score(sub, col, hist), "scope": f"top {N_PROPHET} series"})
res = pd.DataFrame(res).round(4); res.to_csv(OUT_DIR / "metrics_baselines.csv", index=False); res

,model,n_series,WAPE,RMSSE,scope
0,naive,600,0.8764,1.0720,all series
1,snaive,600,0.9299,1.1651,all series
2,ma28,600,0.7037,0.8428,all series
3,naive,40,0.6603,1.1249,top 40 series
4,snaive,40,0.7213,1.2373,top 40 series
5,ma28,40,0.5171,0.8927,top 40 series
6,prophet,40,0.5264,0.8915,top 40 series


**Reading the table:** a good global ML model should clearly beat seasonal naive. Prophet is strong on
smooth, high-volume series but struggles with intermittent items — a key point for the presentation.